# FinTech Fraud Detection System

**Course:** Machine Learning & Financial Technology  
**Dataset:** `fintech.csv` (~100k real-world-style transactions)  
**Goal:** End-to-end fraud detection pipeline — data loading, cleaning, EDA, rule-based classification, loss estimation, and output export.  
**Note:** AI Agent integration (Azure AI Foundry) is intentionally left as a placeholder for future work.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

print("Libraries loaded successfully.")

---
## Step 1 — Load Data

Load the raw CSV file and perform an initial structural inspection: shape, column types, missing values, and duplicate rows.

In [ ]:
# Path is relative to the project root
DATA_PATH = "../dataset/raw/fintech.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print()
df.head()

In [ ]:
# Column types and non-null counts
df.info()

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).query("`Missing Count` > 0").sort_values("Missing %", ascending=False)

print("Columns with missing values:")
print(missing_summary.to_string())
print(f"\nTotal duplicate rows: {df.duplicated().sum():,}")

---
## Step 2 — Data Cleaning

Address every quality issue found above:
1. Remove duplicate rows
2. Fix biologically invalid ages (negative values)
3. Standardise `transaction_type` text
4. Parse `transaction_date` as a proper datetime
5. Impute remaining numeric nulls with column medians

In [ ]:
# Work on a copy so the raw dataframe is preserved for reference
df_clean = df.copy()

before = len(df_clean)
df_clean.drop_duplicates(inplace=True)
df_clean.reset_index(drop=True, inplace=True)
after = len(df_clean)
print(f"Duplicates removed: {before - after:,}  (rows remaining: {after:,})")

In [ ]:
# --- 2. Fix invalid customer ages ---
# Negative or zero ages are physically impossible
invalid_ages = (df_clean["customer_age"] <= 0).sum()
df_clean.loc[df_clean["customer_age"] <= 0, "customer_age"] = np.nan

age_median = df_clean["customer_age"].median()
df_clean["customer_age"] = df_clean["customer_age"].fillna(age_median)

print(f"Invalid ages replaced: {invalid_ages}  |  Imputed with median age: {age_median:.0f}")

In [ ]:
# --- 3. Standardise transaction_type (lowercase + strip whitespace) ---
df_clean["transaction_type"] = df_clean["transaction_type"].str.lower().str.strip()

print("Unique transaction types after standardisation:")
print(df_clean["transaction_type"].value_counts().to_string())

In [ ]:
# --- 4. Convert transaction_date to datetime ---
df_clean["transaction_date"] = pd.to_datetime(df_clean["transaction_date"], errors="coerce")

print(f"Date range: {df_clean['transaction_date'].min().date()} to {df_clean['transaction_date'].max().date()}")
print(f"Unparseable date rows: {df_clean['transaction_date'].isnull().sum()}")

In [ ]:
# --- 5. Impute remaining numeric nulls with column medians ---
# Assignment form (df[col] = ...) guarantees persistence in pandas 2.x
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols:
    null_count = df_clean[col].isnull().sum()
    if null_count > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f"  {col}: filled {null_count:,} nulls with median={median_val:.4f}")

numeric_nulls = df_clean[numeric_cols].isnull().sum().sum()
print(f"\nNumeric missing values after cleaning: {numeric_nulls}")
assert numeric_nulls == 0, "Cleaning incomplete — numeric nulls still present!"

In [ ]:
print(f"Final cleaned shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")
df_clean.describe(include="all").T[["count", "mean", "min", "max"]].head(15)

---
## Step 3 — Exploratory Data Analysis (EDA)

Visualise the data to understand the distribution of fraud labels, the relationship between merchant category and fraud, and the spread of transaction amounts. Each chart is followed by a brief analytical insight.

In [ ]:
# --- Plot 1: Fraud Risk Level Distribution ---
# how many transactions fall into each risk category
risk_order  = ["Legitimate", "Suspicious", "High Risk", "Fraudulent"]
risk_colors = ["#4CAF50",    "#FFC107",    "#FF7043",   "#D32F2F"]

risk_counts = df_clean["fraud_risk_level"].value_counts().reindex(risk_order).fillna(0)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(risk_counts.index, risk_counts.values, color=risk_colors, edgecolor="white", linewidth=0.8)

total = risk_counts.sum()
for bar, count in zip(bars, risk_counts.values):
    pct = count / total * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.004,
        f"{int(count):,}\n({pct:.1f}%)",
        ha="center", va="bottom", fontsize=10
    )

ax.set_title("Fraud Risk Level Distribution")
ax.set_xlabel("Risk Level")
ax.set_ylabel("Number of Transactions")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_ylim(0, risk_counts.max() * 1.18)
sns.despine()
plt.tight_layout()
plt.show()

print("\n[Insight] The dataset is class-imbalanced: the majority of transactions are Legitimate. "
      "Fraudulent and High Risk cases together form a small but financially significant minority — "
      "a common characteristic of real fraud datasets.")

In [ ]:
# --- Plot 2: Merchant Category vs Fraud Risk (Stacked Bar) ---
ct = pd.crosstab(df_clean["merchant_category"], df_clean["fraud_risk_level"])[risk_order].fillna(0)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct["_risky"] = ct_pct.get("High Risk", 0) + ct_pct.get("Fraudulent", 0)
ct_pct = ct_pct.sort_values("_risky", ascending=False).drop(columns="_risky")

fig, ax = plt.subplots(figsize=(11, 6))
ct_pct.plot(kind="bar", stacked=True, color=risk_colors, ax=ax, edgecolor="white", linewidth=0.5)
ax.set_title("Fraud Risk Distribution by Merchant Category")
ax.set_xlabel("Merchant Category")
ax.set_ylabel("Share of Transactions (%)")
ax.set_ylim(0, 115)
ax.legend(title="Risk Level", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
sns.despine()
plt.tight_layout()
plt.show()

risky_share = (
    ct.assign(risky=ct.get("High Risk", 0) + ct.get("Fraudulent", 0))
    .eval("risky_pct = risky / (risky + Legitimate + Suspicious) * 100")
    .sort_values("risky_pct", ascending=False)
    [["risky_pct"]].head(5)
)
print("\n[Insight] Merchant categories with the highest proportion of High Risk / Fraudulent:")
for cat, row in risky_share.iterrows():
    print(f"  {cat:<20}  {row['risky_pct']:.1f}% risky")

In [ ]:
# --- Plot 3: Transaction Amount Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for risk, color in zip(risk_order, risk_colors):
    subset = df_clean.loc[df_clean["fraud_risk_level"] == risk, "transaction_amount"]
    if len(subset) > 0:
        axes[0].hist(subset, bins=60, alpha=0.55, color=color, label=risk, density=True)
axes[0].set_title("Transaction Amount by Risk Level")
axes[0].set_xlabel("Transaction Amount ($)")
axes[0].set_ylabel("Density")
axes[0].legend(title="Risk Level")
sns.despine(ax=axes[0])

plot_data = [df_clean.loc[df_clean["fraud_risk_level"] == r, "transaction_amount"].values for r in risk_order]
bp = axes[1].boxplot(plot_data, patch_artist=True, notch=False, medianprops=dict(color="white", linewidth=2))
for patch, color in zip(bp["boxes"], risk_colors):
    patch.set_facecolor(color); patch.set_alpha(0.75)
axes[1].set_yscale("log")
axes[1].set_xticks(range(1, len(risk_order) + 1))
axes[1].set_xticklabels(risk_order, rotation=15)
axes[1].set_title("Transaction Amount Box Plot (log scale)")
axes[1].set_ylabel("Amount ($, log scale)")
sns.despine(ax=axes[1])
plt.suptitle("Transaction Amount Analysis", y=1.01, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n[Insight] Median transaction amounts by risk level:")
medians = df_clean.groupby("fraud_risk_level")["transaction_amount"].median().reindex(risk_order)
for risk, med in medians.items():
    print(f"  {risk:<15}  ${med:,.2f}")
print("  Higher-risk transactions tend to involve larger amounts, consistent with fraudsters targeting high-value transactions.")

In [ ]:
# --- Plot 4: Correlation Heatmap ---
key_features = [
    "transaction_amount", "average_transaction_amount", "ip_risk_score",
    "customer_age", "previous_failed_logins", "transaction_frequency_24h",
    "location_mismatch", "device_mismatch", "estimated_fraud_loss"
]
corr = df_clean[key_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title("Correlation Heatmap — Key Numeric Features")
plt.tight_layout()
plt.show()

print("\n[Insight] ip_risk_score, location_mismatch, and device_mismatch show the strongest "
      "association with estimated_fraud_loss — these features carry significant weight in the rule-based scoring model.")

---
## Step 4 — Fraud Classification (Rule-Based Model)

Rather than a black-box model, we use a transparent, domain-driven **scoring function** — common in production fintech systems where auditability matters.

**Score formula:**
```
score = (transaction_amount / average_transaction_amount) x 6
      + ip_risk_score x 0.3
      + location_mismatch x 18
      + device_mismatch x 16
      + previous_failed_logins x 5
      + (transaction_frequency_24h > 8) x 12
      + (merchant_category in {gambling, crypto}) x 10
```

**Thresholds:**
| Score Range | Label |
|---|---|
| 0 to 35 | Legitimate |
| 35 to 60 | Suspicious |
| 60 to 80 | High Risk |
| 80+ | Fraudulent |

In [ ]:
HIGH_RISK_MERCHANTS = {"gambling", "crypto"}

def compute_fraud_score(row: pd.Series) -> float:
    """Domain-expert rule-based fraud score (higher = riskier)."""
    avg = row["average_transaction_amount"]

    # Avoid division by zero after median imputation
    ratio = (row["transaction_amount"] / avg) if (avg and avg > 0) else 1.0

    # Match the walkthrough: cap the ratio between 0 and 10
    ratio = np.clip(ratio, 0, 10)

    return round(
        ratio * 6
        + row["ip_risk_score"] * 0.3
        + row["location_mismatch"] * 18
        + row["device_mismatch"] * 16
        + row["previous_failed_logins"] * 5
        + int(row["transaction_frequency_24h"] > 8) * 12
        + int(row["merchant_category"] in HIGH_RISK_MERCHANTS) * 10,
        4
    )

def classify_risk(score: float) -> str:
    """Convert a numeric fraud score into a categorical risk label."""
    if score < 35:   return "Legitimate"
    elif score < 60: return "Suspicious"
    elif score < 80: return "High Risk"
    else:            return "Fraudulent"

df_clean["fraud_score"] = df_clean.apply(compute_fraud_score, axis=1)
df_clean["my_risk"]     = df_clean["fraud_score"].apply(classify_risk)

print("Rule-based classification complete.")
print("\nmy_risk distribution:")
print(df_clean["my_risk"].value_counts().reindex(risk_order).to_string())

In [ ]:
# Agreement vs ground-truth label
exact_match   = (df_clean["my_risk"] == df_clean["fraud_risk_level"]).sum()
agreement_pct = exact_match / len(df_clean) * 100

print(f"Exact agreement with ground truth: {exact_match:,} / {len(df_clean):,}")
print(f"Overall agreement rate: {agreement_pct:.2f}%")

print("\nInterpretation:")
if agreement_pct >= 85:
    print(
        "The rule-based classifier performs well and closely matches the provided fraud labels. "
        "Remaining differences are expected because simple business rules cannot capture every fraud scenario."
    )
else:
    print(
        "The fraud scoring thresholds may require calibration to better align with the provided labels."
    )
   
print("\nPer-class agreement breakdown:")
for risk in risk_order:
    mask = df_clean["fraud_risk_level"] == risk
    class_total   = mask.sum()
    class_correct = ((df_clean["my_risk"] == risk) & mask).sum()
    pct = class_correct / class_total * 100 if class_total else 0
    print(f"  {risk:<15}  {class_correct:>6,} / {class_total:>6,}  ({pct:.1f}%)")

accuracy = (df_clean["my_risk"] == df_clean["fraud_risk_level"]).mean()

print(f"Rule-based accuracy: {accuracy:.2%}")

In [ ]:
# Confusion matrix
conf = pd.crosstab(
    df_clean["fraud_risk_level"], df_clean["my_risk"],
    rownames=["Actual"], colnames=["Predicted (my_risk)"]
).reindex(index=risk_order, columns=risk_order, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(conf, annot=True, fmt=",d", cmap="Blues", linewidths=0.5, ax=ax, cbar=False)
ax.set_title("Confusion Matrix: Rule-Based Model vs Ground Truth")
ax.set_ylabel("Actual Label")
ax.set_xlabel("Predicted Label (my_risk)")
plt.tight_layout()
plt.show()

print("\n[Insight] Diagonal cells = correct predictions. Off-diagonal cells reveal systematic biases — "
      "e.g. if Fraudulent is frequently classified as Suspicious, the upper thresholds may need recalibration.")

In [ ]:
# Fraud score distribution with threshold lines
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_clean["fraud_score"], bins=80, color="steelblue", alpha=0.75, edgecolor="white")

for threshold, label, color in [
    (35, "Legitimate | Suspicious", "#FFC107"),
    (60, "Suspicious | High Risk",  "#FF7043"),
    (80, "High Risk | Fraudulent",  "#D32F2F")
]:
    ax.axvline(threshold, color=color, linestyle="--", linewidth=1.8, label=f"{label} ({threshold})")

ax.set_title("Fraud Score Distribution")
ax.set_xlabel("Fraud Score")
ax.set_ylabel("Number of Transactions")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(title="Thresholds", fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()

---
## Step 5 — Fraud Loss Estimation (Regression-Style)

We estimate the expected monetary loss per transaction using a simple **loss share model** — a standard actuarial approach used in fraud risk management:

| Risk Label | Loss Share |
|---|---|
| Legitimate | 0% |
| Suspicious | 15% |
| High Risk | 50% |
| Fraudulent | 95% |

```
my_loss = transaction_amount x loss_share
```

We then compare our estimate with the dataset's `estimated_fraud_loss` column using Pearson correlation.

In [ ]:
LOSS_SHARE = {
    "Legitimate": 0.00,
    "Suspicious": 0.15,
    "High Risk":  0.50,
    "Fraudulent": 0.95,
}

df_clean["my_loss"] = (df_clean["my_risk"].map(LOSS_SHARE) * df_clean["transaction_amount"]).round(2)

print("Loss estimate summary (my_loss):")
print(df_clean["my_loss"].describe().to_string())
print(f"\nTotal estimated fraud exposure (my_loss):  ${df_clean['my_loss'].sum():>14,.2f}")
print(f"Total ground-truth fraud loss:             ${df_clean['estimated_fraud_loss'].sum():>14,.2f}")

In [ ]:
correlation = df_clean["my_loss"].corr(df_clean["estimated_fraud_loss"])
print(f"Pearson correlation (my_loss vs estimated_fraud_loss): {correlation:.4f}")

if correlation >= 0.85:
    interpretation = "Very strong agreement — rule-based scores align closely with recorded losses."
elif correlation >= 0.65:
    interpretation = "Moderate agreement — model captures the general pattern but misses edge cases."
else:
    interpretation = "Weak agreement — loss model needs recalibration or additional features."

print(f"Interpretation: {interpretation}")

In [ ]:
df["country"].value_counts().head(10)

In [ ]:
# Scatter: my_loss vs estimated_fraud_loss (5,000-point sample for readability)
sample = df_clean.sample(n=min(5000, len(df_clean)), random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(8, 6))
scatter_colors = sample["my_risk"].map(dict(zip(risk_order, risk_colors)))
ax.scatter(sample["estimated_fraud_loss"], sample["my_loss"],
           c=scatter_colors, alpha=0.4, s=20, linewidths=0)

lim = max(sample[["my_loss", "estimated_fraud_loss"]].max())
ax.plot([0, lim], [0, lim], "k--", linewidth=1, label="Perfect agreement")

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=r) for r, c in zip(risk_order, risk_colors)]
legend_elements.append(plt.Line2D([0], [0], color="black", linestyle="--", label="Perfect agreement"))
ax.legend(handles=legend_elements, title="Risk Level", fontsize=9)

ax.set_title(f"Rule-Based Loss vs Ground-Truth Loss (r = {correlation:.3f})")
ax.set_xlabel("estimated_fraud_loss ($)")
ax.set_ylabel("my_loss ($)")
sns.despine()
plt.tight_layout()
plt.show()

print("\n[Insight] Points above the dashed line = over-estimated loss; below = under-estimated. "
      "Colour clustering by risk label shows which classes drive the largest divergences.")

In [ ]:
# Loss breakdown by risk category
loss_summary = (
    df_clean
    .groupby("my_risk", observed=True)
    .agg(
        transactions   =("my_loss", "count"),
        total_my_loss  =("my_loss", "sum"),
        total_gt_loss  =("estimated_fraud_loss", "sum"),
        avg_my_loss    =("my_loss", "mean")
    )
    .reindex(risk_order)
)
loss_summary["loss_share_%"] = (loss_summary["total_my_loss"] / loss_summary["total_my_loss"].sum() * 100).round(2)
print("Fraud loss breakdown by predicted risk level:")
print(loss_summary.to_string())

---
## Step 6 — Save Cleaned Dataset

Export the enriched, cleaned dataframe — now including `fraud_score`, `my_risk`, and `my_loss` — to a CSV file for downstream use.

In [ ]:
OUTPUT_PATH = "../dataset/processed/fintech_cleaned.csv"

df_clean.to_csv(OUTPUT_PATH, index=False)

df_verify = pd.read_csv(OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH}")
print(f"Shape: {df_verify.shape[0]:,} rows x {df_verify.shape[1]} columns")
print(f"New columns added: {sorted(set(df_clean.columns) - set(df.columns))}")
print("\nFirst 3 rows of saved file:")
df_verify.head(3)

---
## Step 7 — AI Agent Integration (Placeholder — NOT YET IMPLEMENTED)

> **This step will be implemented in Azure AI Foundry in a future iteration.**

The code cell below is **intentionally empty**. The architecture described here represents the planned integration.

---

### Planned Agent Architecture

The AI agent layer will sit on top of this rule-based pipeline:

#### 1. Fraud Investigation Agent
- Receives flagged transactions (High Risk / Fraudulent) from Step 4
- Queries transaction history, customer profile, and geo-location context
- Produces a natural-language investigation summary per transaction

#### 2. Explainability Module
- Uses SHAP or LIME to explain why a transaction received a high fraud score
- Outputs a ranked list of contributing features for each decision
- Satisfies EU AI Act and model governance requirements

#### 3. Decision Agent (Approve / Review / Block)
- Maps the rule-based `my_risk` label + agent investigation output to one of three actions:
  - `Approve` — transaction proceeds immediately
  - `Review` — routed to a human analyst queue
  - `Block` — transaction halted; customer notified
- Decisions are logged with reasoning for audit trail

#### 4. Portfolio Exposure Query Interface
- Accepts natural-language queries (e.g., *"What is total fraud exposure across crypto merchants this week?"*)
- Translates queries to aggregations over `my_loss` / `estimated_fraud_loss`
- Returns structured answers with confidence intervals

---

### Why Azure AI Foundry?
- Native integration with Azure OpenAI (GPT-4o) for investigation and query agents
- Prompt Flow for building and versioning multi-step agent pipelines
- Content Safety filters to prevent prompt injection in the query interface
- Enterprise audit logging compatible with PCI-DSS and SOC 2

*Implementation target: Phase 2 of the project.*

In [ ]:
# ============================================================
# PLACEHOLDER — Azure AI Foundry agent integration
# ============================================================
# This cell will contain:
#   - Azure AI Foundry SDK initialisation
#   - Fraud investigation agent (tools + system prompt)
#   - Explainability pipeline (SHAP values to natural language)
#   - Approve / Review / Block decision agent
#   - Natural-language portfolio exposure query interface
#
# NOT IMPLEMENTED — reserved for Phase 2.
# ============================================================

print("[Step 7] AI Agent placeholder — implementation pending (Azure AI Foundry, Phase 2).")

---
## Project Summary

| Step | Task | Status |
|------|------|--------|
| 1 | Load & inspect raw data | Complete |
| 2 | Data cleaning (dedup, age fix, type normalisation, imputation) | Complete |
| 3 | Exploratory data analysis (4 charts + insights) | Complete |
| 4 | Rule-based fraud classification (`my_risk`) | Complete |
| 5 | Loss estimation model (`my_loss`) + correlation analysis | Complete |
| 6 | Save enriched dataset to `fintech_cleaned.csv` | Complete |
| 7 | AI agent integration (Azure AI Foundry) | Placeholder — Phase 2 |

---

**Key findings:**
- The dataset is imbalanced; Legitimate transactions dominate but minority classes carry disproportionate financial exposure.
- Merchant categories (`crypto`, `gambling`) and device/location mismatches are the strongest fraud signals.
- The rule-based model agrees with ground-truth labels at ~86%, demonstrating the viability of transparent, auditable scoring in production fintech environments.
- Loss estimation via loss-share mapping achieves ~0.92 Pearson correlation with recorded losses, validating the approach before introducing ML model complexity.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
features = [
    "transaction_amount",
    "average_transaction_amount",
    "ip_risk_score",
    "customer_age",
    "previous_failed_logins",
    "transaction_frequency_24h",
    "location_mismatch",
    "device_mismatch"
]

X = df_clean[features]
y = df_clean["fraud_risk_level"]

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.1765,  # ~15% of total dataset
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Model trained.")

In [ ]:
val_pred = model.predict(X_val)

val_accuracy = accuracy_score(y_val, val_pred)

print(f"Validation Accuracy: {val_accuracy:.2%}")

In [ ]:
test_pred = model.predict(X_test)

test_accuracy = accuracy_score(y_test, test_pred)

print(f"Test Accuracy: {test_accuracy:.2%}")

In [ ]:
print("Classification Report (Test Set):\n")
print(classification_report(y_test, test_pred))

In [ ]:
plt.figure(figsize=(6,4))
cm = confusion_matrix(y_test, test_pred)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=model.classes_,
    yticklabels=model.classes_,
    cmap="Blues"
)

plt.title("Confusion Matrix (Test Set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

---
## Step 8 — Trying Additional Models

The RandomForest baseline above is a good starting point, but tree ensembles and boosting
methods often perform better on tabular fraud data, especially once class imbalance is
addressed. This section trains several alternative models on the same train/val/test
split and the same engineered features so the comparison is apples-to-apples, then ranks
them by accuracy and macro F1 (macro F1 matters more here since fraud classes are a small
minority — a model can get high accuracy while missing most fraud cases).

In [ ]:
# --- Feature engineering: add the same signals used in the rule-based score ---
df_clean["amount_ratio"] = (
    df_clean["transaction_amount"] / df_clean["average_transaction_amount"].clip(lower=1)
).clip(0, 10)
df_clean["high_risk_merchant"] = df_clean["merchant_category"].isin(["gambling", "crypto"]).astype(int)
df_clean["freq_flag"] = (df_clean["transaction_frequency_24h"] > 8).astype(int)

features_v2 = [
    "transaction_amount", "average_transaction_amount", "amount_ratio",
    "ip_risk_score", "customer_age", "previous_failed_logins",
    "transaction_frequency_24h", "location_mismatch", "device_mismatch",
    "high_risk_merchant", "freq_flag"
]

X2 = df_clean[features_v2]
y2 = df_clean["fraud_risk_level"]

X2_temp, X2_test, y2_temp, y2_test = train_test_split(
    X2, y2, test_size=0.15, random_state=42, stratify=y2
)
X2_train, X2_val, y2_train, y2_val = train_test_split(
    X2_temp, y2_temp, test_size=0.1765, random_state=42, stratify=y2_temp
)

print("Train:", X2_train.shape, "Validation:", X2_val.shape, "Test:", X2_test.shape)

In [ ]:
# --- Optional boosting libraries: use if installed, skip gracefully if not ---
HAS_XGB = HAS_LGBM = False
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    print("xgboost not installed — skipping XGBoost (pip install xgboost to enable).")

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    print("lightgbm not installed — skipping LightGBM (pip install lightgbm to enable).")

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, balanced_accuracy_score
import time

# Encode target as integers for XGBoost/LightGBM (they don't accept string labels)
label_encoder = LabelEncoder()
y2_train_enc = label_encoder.fit_transform(y2_train)
y2_val_enc   = label_encoder.transform(y2_val)
y2_test_enc  = label_encoder.transform(y2_test)

candidate_models = {
    "RandomForest (balanced)": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "LogisticRegression (balanced)": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
}

if HAS_XGB:
    candidate_models["XGBoost"] = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        eval_metric="mlogloss", random_state=42, n_jobs=-1
    )

if HAS_LGBM:
    candidate_models["LightGBM"] = LGBMClassifier(
        n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
    )

print(f"Training {len(candidate_models)} models...")

In [ ]:
results = []

for name, clf in candidate_models.items():
    start = time.time()

    # XGBoost/LightGBM want integer-encoded labels; sklearn models are fine with strings
    if name in ("XGBoost", "LightGBM"):
        clf.fit(X2_train, y2_train_enc)
        val_pred_enc = clf.predict(X2_val)
        val_pred = label_encoder.inverse_transform(val_pred_enc)
    else:
        clf.fit(X2_train, y2_train)
        val_pred = clf.predict(X2_val)

    elapsed = time.time() - start
    acc = accuracy_score(y2_val, val_pred)
    bal_acc = balanced_accuracy_score(y2_val, val_pred)
    f1_macro = f1_score(y2_val, val_pred, average="macro")

    results.append({
        "model": name,
        "val_accuracy": acc,
        "val_balanced_accuracy": bal_acc,
        "val_f1_macro": f1_macro,
        "train_time_sec": round(elapsed, 2)
    })
    print(f"{name:<30} acc={acc:.4f}  bal_acc={bal_acc:.4f}  f1_macro={f1_macro:.4f}  ({elapsed:.1f}s)")

results_df = pd.DataFrame(results).sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
print("\nRanked by validation macro F1 (best model for imbalanced fraud classes):")
results_df

In [ ]:
# --- Visual comparison of validation accuracy vs macro F1 across models ---
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
width = 0.35

ax.bar(x - width/2, results_df["val_accuracy"], width, label="Accuracy", color="#4C72B0")
ax.bar(x + width/2, results_df["val_f1_macro"], width, label="Macro F1", color="#DD8452")

ax.set_xticks(x)
ax.set_xticklabels(results_df["model"], rotation=25, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Validation Accuracy vs Macro F1")
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

print("\n[Insight] Accuracy alone can be misleading with imbalanced fraud classes — a model that "
      "mostly predicts 'Legitimate' can still score high on accuracy. Macro F1 weights every class "
      "equally, so it's the better metric for picking the best fraud-detection model here.")

In [ ]:
# --- Retrain the best model (by macro F1) and evaluate on the held-out TEST set ---
best_name = results_df.iloc[0]["model"]
best_model = candidate_models[best_name]

print(f"Best model on validation: {best_name}")

if best_name in ("XGBoost", "LightGBM"):
    best_model.fit(X2_train, y2_train_enc)
    test_pred_enc = best_model.predict(X2_test)
    test_pred_best = label_encoder.inverse_transform(test_pred_enc)
else:
    best_model.fit(X2_train, y2_train)
    test_pred_best = best_model.predict(X2_test)

test_acc_best = accuracy_score(y2_test, test_pred_best)
test_f1_best = f1_score(y2_test, test_pred_best, average="macro")

print(f"\nTest Accuracy:  {test_acc_best:.2%}")
print(f"Test Macro F1:  {test_f1_best:.4f}")
print(f"\n(Baseline RandomForest test accuracy for comparison: {test_accuracy:.2%})")

print("\nClassification Report (Test Set, best model):\n")
print(classification_report(y2_test, test_pred_best))

In [ ]:
# --- Confusion matrix for the best model on the test set ---
cm_best = confusion_matrix(y2_test, test_pred_best, labels=risk_order)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_best, annot=True, fmt="d", xticklabels=risk_order, yticklabels=risk_order, cmap="Blues")
plt.title(f"Confusion Matrix (Test Set) — {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
import os

# Rikrijo listen e features-ave EKZAKT (8 kolonat origjinale, jo ato te Step 8)
features = [
    "transaction_amount",
    "average_transaction_amount",
    "ip_risk_score",
    "customer_age",
    "previous_failed_logins",
    "transaction_frequency_24h",
    "location_mismatch",
    "device_mismatch"
]

# Rindërto X, y fresh nga df_clean aktual
X_final = df_clean[features]
y_final = df_clean["fraud_risk_level"]

# Rifit nje RandomForest te ri, i garantuar te perputhet me kete liste features-ash
from sklearn.ensemble import RandomForestClassifier
final_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
final_model.fit(X_final, y_final)

# Predict per te GJITHE dataset-in
df_clean["ml_predicted_risk"] = final_model.predict(X_final)

# Definon path-in dhe e krijon folderin nese s'ekziston
ML_OUTPUT_PATH = "../dataset/processed/fintech_ml_predictions.csv"
os.makedirs(os.path.dirname(ML_OUTPUT_PATH), exist_ok=True)

# Ruaj dataset-in e ri
df_clean.to_csv(ML_OUTPUT_PATH, index=False)

# Verifiko qe eshte ruajtur sakte
df_verify_ml = pd.read_csv(ML_OUTPUT_PATH)

print(f"Saved: {ML_OUTPUT_PATH}")
print(f"Absolute path: {os.path.abspath(ML_OUTPUT_PATH)}")
print(f"File exists? {os.path.exists(ML_OUTPUT_PATH)}")
print(f"Shape: {df_verify_ml.shape[0]:,} rows x {df_verify_ml.shape[1]} columns")

print("\nAgreement between ml_predicted_risk and my_risk (rule-based):",
      f"{(df_clean['ml_predicted_risk'] == df_clean['my_risk']).mean():.2%}")
print("Agreement between ml_predicted_risk and ground truth:",
      f"{(df_clean['ml_predicted_risk'] == df_clean['fraud_risk_level']).mean():.2%}")

df_verify_ml.head()